# SELECT and Filtering — SQLite Demo

Run these examples to practice selecting columns, filtering, matching text, handling NULL, sorting, and limiting results.

In [1]:
import sqlite3
import pandas as pd
con = sqlite3.connect(':memory:')
print(sqlite3.sqlite_version)

3.46.1


## 1. Create and populate sample tables

In [2]:
con.executescript("""
CREATE TABLE students (
  student_id INTEGER PRIMARY KEY,
  student_name TEXT NOT NULL,
  department TEXT,
  age INTEGER,
  email TEXT
);
CREATE TABLE products (
  product_id INTEGER PRIMARY KEY,
  product_name TEXT NOT NULL,
  category TEXT,
  price REAL,
  product_code TEXT
);
CREATE TABLE orders (
  order_id INTEGER PRIMARY KEY,
  customer_id INTEGER,
  order_date TEXT,
  status TEXT,
  total REAL
);
INSERT INTO students VALUES
(1, 'Asha', 'Data Science', 20, 'asha@example.com'),
(2, 'Ravi', 'Computer Science', 22, NULL),
(3, 'Anaya', 'Data Science', 19, 'anaya@example.com'),
(4, 'Kabir', 'Cybersecurity', 25, 'kabir@example.com'),
(5, 'Mina', NULL, 27, 'mina@example.com');
INSERT INTO products VALUES
(1, 'Laptop Pro', 'Laptop', 950, 'LP-100'),
(2, 'Smartphone', 'Phone', 650, 'PH-200'),
(3, 'Tablet Mini', 'Tablet', 300, 'TB-301'),
(4, 'USB Cable', 'Accessories', 15, 'ABX7'),
(5, 'Headphones', 'Accessories', 120, 'HP-500'),
(6, 'Budget Phone', 'Phone', 180, 'PH-101');
INSERT INTO orders VALUES
(101, 1, '2026-01-05', 'pending', 250),
(102, 2, '2026-01-18', 'completed', 1200),
(103, 3, '2026-02-02', 'processing', 400),
(104, 4, '2025-12-31', 'completed', 75);
""")
print("Sample tables created")

Sample tables created


## 2. Select columns and create an expression

In [3]:
pd.read_sql_query("""
SELECT student_id, student_name AS name, age
FROM students
ORDER BY student_id
""", con)

,student_id,name,age
0,1,Asha,20
1,2,Ravi,22
2,3,Anaya,19
3,4,Kabir,25
4,5,Mina,27


In [4]:
pd.read_sql_query("""
SELECT product_name, price, ROUND(price * 0.90, 2) AS discounted_price
FROM products
ORDER BY price DESC
""", con)

,product_name,price,discounted_price
0,Laptop Pro,950.0,855.0
1,Smartphone,650.0,585.0
2,Tablet Mini,300.0,270.0
3,Budget Phone,180.0,162.0
4,Headphones,120.0,108.0
5,USB Cable,15.0,13.5


## 3. Filter with comparisons and logical conditions

In [5]:
pd.read_sql_query("""
SELECT product_name, price
FROM products
WHERE price >= 100 AND price < 700
ORDER BY price
""", con)

,product_name,price
0,Headphones,120.0
1,Budget Phone,180.0
2,Tablet Mini,300.0
3,Smartphone,650.0


In [6]:
pd.read_sql_query("""
SELECT student_name, department, age
FROM students
WHERE (department = 'Data Science' OR department = 'Cybersecurity')
  AND age BETWEEN 18 AND 25
ORDER BY student_name
""", con)

,student_name,department,age
0,Anaya,Data Science,19
1,Asha,Data Science,20
2,Kabir,Cybersecurity,25


## 4. IN, LIKE, and DISTINCT

In [7]:
pd.read_sql_query("""
SELECT product_name, category
FROM products
WHERE category IN ('Laptop', 'Tablet')
""", con)

,product_name,category
0,Laptop Pro,Laptop
1,Tablet Mini,Tablet


In [8]:
pd.read_sql_query("""
SELECT product_name, product_code
FROM products
WHERE product_code LIKE 'AB_7'
""", con)

,product_name,product_code
0,USB Cable,ABX7


In [9]:
pd.read_sql_query("SELECT DISTINCT department FROM students ORDER BY department", con)

,department
0,None
1,Computer Science
2,Cybersecurity
3,Data Science


## 5. NULL checks and fallback display

In [10]:
pd.read_sql_query("SELECT student_name FROM students WHERE email IS NULL", con)

,student_name
0,Ravi


In [11]:
pd.read_sql_query("""
SELECT student_name, COALESCE(email, 'Not provided') AS email_display
FROM students
ORDER BY student_id
""", con)

,student_name,email_display
0,Asha,asha@example.com
1,Ravi,Not provided
2,Anaya,anaya@example.com
3,Kabir,kabir@example.com
4,Mina,mina@example.com


## 6. Date filtering with a half-open interval

In [12]:
pd.read_sql_query("""
SELECT order_id, order_date, total
FROM orders
WHERE order_date >= '2026-01-01'
  AND order_date < '2026-02-01'
ORDER BY order_date
""", con)

,order_id,order_date,total
0,101,2026-01-05,250.0
1,102,2026-01-18,1200.0


## 7. ORDER BY and LIMIT

In [13]:
pd.read_sql_query("""
SELECT product_name, price
FROM products
ORDER BY price DESC, product_name ASC
LIMIT 3
""", con)

,product_name,price
0,Laptop Pro,950.0
1,Smartphone,650.0
2,Tablet Mini,300.0


## 8. Parameterized query in Python

In [14]:
department_value = "Data Science"
pd.read_sql_query(
    "SELECT student_id, student_name FROM students WHERE department = ?",
    con,
    params=(department_value,)
)

,student_id,student_name
0,1,Asha
1,3,Anaya


## 9. Inspect and close

In [15]:
summary = pd.read_sql_query("SELECT status, COUNT(*) AS order_count FROM orders GROUP BY status ORDER BY status", con)
summary

,status,order_count
0,completed,2
1,pending,1
2,processing,1


In [16]:
con.close()
print("SQLite connection closed.")

SQLite connection closed.
